# Absorption spectrum to X-ray refractive index

This notebook converts an absolute absorption measurement to $\beta(E)$, extends it with a local Henke/CXRO dataset, and calculates $\delta(E)$ for

$$n(E) = 1 - \delta(E) + i\beta(E).$$

It performs no network access. Download the appropriate Henke/CXRO optical-constant text file separately.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from library import kramers_kronig as kk

## Input data

The example expects `absorption_spectrum.npz` to contain `energy_ev` and one absorption array. Select the matching `input_kind`:

- `absorption_coefficient`: linear attenuation coefficient $\mu$ in m$^{-1}$
- `optical_depth`: $-\ln(I/I_0)=\mu t$
- `transmission`: $I/I_0$

Thickness is required for optical depth or transmission. Arbitrary-unit XAS requires a separate absolute calibration before this conversion.

In [ ]:
absorption_file = Path("data/absorption_spectrum.npz")
henke_file = Path("data/henke_optical_constants.txt")

input_kind = "optical_depth"  # or "transmission", "absorption_coefficient"
thickness_m = 100e-9

absorption_data = np.load(absorption_file)
energy_ev = absorption_data["energy_ev"]
absorption = absorption_data["absorption"]

beta_measured = kk.absorption_to_beta(
    energy_ev,
    absorption,
    input_kind=input_kind,
    thickness_m=thickness_m,
)

## Extend with Henke/CXRO data

The default loader assumes columns `energy_eV, delta, beta`. Change the column indices if your text export differs. Measured and reference values must both be absolute $\beta$.

In [ ]:
henke_energy_ev, henke_delta, henke_beta = kk.load_henke_refractive_index(
    henke_file,
    energy_column=0,
    delta_column=1,
    beta_column=2,
    skiprows=0,
)

(
    beta_measured,
    delta_measured,
    extended_energy_ev,
    extended_beta,
    beta_correction,
) = kk.refractive_index_from_beta_with_reference(
    measured_energy_ev=energy_ev,
    measured_beta=beta_measured,
    reference_energy_ev=henke_energy_ev,
    reference_delta=henke_delta,
    reference_beta=henke_beta,
    transition_width_ev=2.0,
    return_extended=True,
)

## Calculate the real refractive-index component

The calculation keeps Henke $\delta$ as the broad baseline and KK-transforms only the localized difference $\beta_{measured}-\beta_{Henke}$. This is the near-edge correction strategy described by Watts and avoids truncating the absolute KK integral at the limits of the Henke table.

In [ ]:
henke_beta_at_measurement = np.interp(energy_ev, henke_energy_ev, henke_beta)
henke_delta_at_measurement = np.interp(energy_ev, henke_energy_ev, henke_delta)

fig, axes = plt.subplots(2, 1, figsize=(8, 7), sharex=True)
axes[0].plot(energy_ev, henke_beta_at_measurement, "--", label=r"Henke $\beta$")
axes[0].plot(energy_ev, beta_measured, "o-", label=r"corrected $\beta(E)$")
axes[1].plot(energy_ev, henke_delta_at_measurement, "--", label=r"Henke $\delta$")
axes[1].plot(energy_ev, delta_measured, "o-", label=r"corrected $\delta(E)$")
axes[0].set_ylabel(r"$\beta$")
axes[1].set_ylabel(r"$\delta$")
axes[1].set_xlabel("Photon energy (eV)")
for ax in axes:
    ax.grid(True)
    ax.legend()
plt.show()

In [ ]:
output_file = Path("data/refractive_index_constraints.npz")
output_file.parent.mkdir(parents=True, exist_ok=True)
np.savez(
    output_file,
    energy_ev=energy_ev,
    beta=beta_measured,
    delta=delta_measured,
    extended_energy_ev=extended_energy_ev,
    extended_beta=extended_beta,
    beta_correction=beta_correction,
)
print(f"Saved {output_file}")